# Module 1 — Part 2: Agents

Lessons 13–14: Function calling + Agentic loop.

**Key shift:** Fixed RAG always searches once with the exact user query.
An agent lets the LLM decide when to search, what to search for, and when to stop.

**Groq adaptation note:**
- Lesson uses OpenAI Responses API: `response.output`, `item.type == 'function_call'`, `function_call_output` message type
- Groq chat.completions: `response.choices[0].message.tool_calls`, tool results use `role: 'tool'`

## Setup

In [ ]:
from dotenv import load_dotenv
import os
import json
load_dotenv()

from openai import OpenAI
from ingest import load_faq_data, build_index

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

documents = load_faq_data()
index = build_index(documents)
print("Index ready")

## Lesson 13 — Function Calling

Without tools the LLM answers from general knowledge — vague and unhelpful for course questions.

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
)

print(response.choices[0].message.content)
# Generic answer — no course-specific knowledge

### Define the search tool

We describe `search` in JSON schema. The LLM reads the `description` to decide when to call it.

In [ ]:
def search(query):
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

# Tool schema — chat.completions format (nested under 'function' key)
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [ ]:
# Send question WITH the tool — LLM decides to call search instead of answering directly
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool],
)

assistant_message = response.choices[0].message
print("Tool calls:", assistant_message.tool_calls)
# Model returns a tool_call instead of a text answer

In [ ]:
# Execute the function and send result back
tool_call = assistant_message.tool_calls[0]
func_name = tool_call.function.name
args = json.loads(tool_call.function.arguments)

print(f"Function: {func_name}, Args: {args}")
# Note: model often rewrites the query to better keywords

results = search(**args)
result_json = json.dumps(results, indent=2)

# Build updated message history
# 1. append the assistant's tool-call message
# 2. append the tool result with role='tool'
messages.append(assistant_message)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json,
})

# Second API call — model now has the search results and generates final answer
response2 = groq_client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[search_tool],
)

print(response2.choices[0].message.content)

## Lesson 14 — The Agentic Loop

Single function call works for one turn. The loop handles:
- Multiple searches per question
- Model recovering from bad results (e.g. typos)
- Unknown number of tool calls in advance

In [ ]:
def make_call(tool_call):
    """Execute a tool call and return the tool result message."""
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result, indent=2),
    }

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, off-topic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
def agent_loop(instructions, question, model=MODEL) -> str:
    messages = [
        {"role": "system", "content": instructions},  # 'developer' in lesson -> 'system' for chat.completions
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")

        response = groq_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        has_tool_calls = bool(assistant_message.tool_calls)

        if has_tool_calls:
            for tool_call in assistant_message.tool_calls:
                print(f"  function_call: {tool_call.function.name} {tool_call.function.arguments}")
                call_output = make_call(tool_call)
                messages.append(call_output)
        else:
            last_answer = assistant_message.content
            print("ASSISTANT:")
            print(last_answer)

        it += 1
        if not has_tool_calls:
            break

    return last_answer

In [ ]:
# Classic typo test — agent searches 'Olama', gets bad results,
# then corrects to 'Ollama' on its own
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

In [ ]:
# Off-topic guardrail test — agent should refuse
agent_loop(instructions, "What's the Queen's Gambit?")